<h3>Final Refit and Single Held-Out Evaluation</h3>


<h3>1. Runtime Setup and Artifact Paths</h3>

<p>The next cell follows the model-training notebook's Colab setup: mount Drive, load the same environment-file locations, and locate or clone the configured repository branch. Colab inputs are:</p>

<p>MyDrive/capstone-data/feature_engineering/: supervised H1/H2 tables and saved fold definitions.</p>

<p>MyDrive/capstone-data/model_training/: tuning_h1.json and tuning_h2.json, including feature order, RF/XGBoost parameters, and Ridge alpha.</p>

<p>Local execution uses the equivalent results/artifacts/ directories. Output directories and horizon configurations are defined here, but no training or test inference is performed.</p>

In [1]:
import base64
import os
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/RAM-AS-RAP/ucs35-mudibo-ml.git'
REPO_BRANCH = 'feature/model-training'

if IN_COLAB:
    drive.mount('/content/drive')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'pandas<3', 'numpy', 'scikit-learn', 'xgboost', 'joblib', 'python-dotenv',
    ])
    from dotenv import load_dotenv
    ENV_PATHS = [
        Path('/content/drive/MyDrive/capstone-data/.env'),
        Path('/content/drive/MyDrive/AgriSence/.env'),
        Path('/content/drive/MyDrive/.env'),
        Path('/content/.env'),
    ]
    for env_path in ENV_PATHS:
        if env_path.exists():
            load_dotenv(env_path, override=False)
    repo_dir = Path('/content/AgriSence_model_training')
    repo_candidates = [repo_dir]
else:
    repo_dir = None
    repo_candidates = [Path.cwd(), *Path.cwd().resolve().parents]

repo_root = next((
    path.resolve() for path in repo_candidates
    if (path / 'pyproject.toml').exists() and (path / 'src/agrisence').exists()
), None)
if repo_root is None and IN_COLAB:
    github_token = os.getenv('GITHUB_TOKEN') or os.getenv('GH_TOKEN')
    if not github_token:
        from google.colab import userdata
        for secret_name in ('GITHUB_TOKEN', 'GH_TOKEN'):
            try:
                github_token = userdata.get(secret_name)
            except Exception:
                continue
            if github_token:
                break
    if not github_token:
        raise RuntimeError(
            'Private repository clone requires GITHUB_TOKEN or GH_TOKEN. '
            'Use Colab Secrets with notebook access enabled, or a Drive .env file.'
        )
    auth_header = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run([
        'git', '-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth_header}',
        'clone', '-b', REPO_BRANCH, REPO_URL, str(repo_dir),
    ], check=True)
    del github_token, auth_header
    repo_root = repo_dir.resolve()
if repo_root is None:
    raise FileNotFoundError('Start this notebook from inside the AgriSence repository.')
if IN_COLAB:
    os.chdir(repo_root)
    load_dotenv(repo_root / '.env', override=False)

import pandas as pd
from IPython.display import display

sys.path.insert(0, str(repo_root / 'src'))

if IN_COLAB:
    feature_dir = Path('/content/drive/MyDrive/capstone-data/feature_engineering')
    tuning_dir = Path('/content/drive/MyDrive/capstone-data/model_training')
else:
    feature_dir = repo_root / 'results/artifacts/feature_engineering'
    tuning_dir = repo_root / 'results/artifacts/model_training'

output_dir = repo_root / 'results/artifacts'
model_dir = output_dir / 'models'
drive_output_dir = Path('/content/drive/MyDrive/capstone-data/final_evaluation') if IN_COLAB else None
drive_model_dir = Path('/content/drive/MyDrive/capstone-data/models') if IN_COLAB else None
horizon_configs = {
    horizon: {
        'data_path': feature_dir / f'model_df_{horizon}.csv',
        'fold_path': feature_dir / f'folds_{horizon}.json',
        'tuning_path': tuning_dir / f'tuning_{horizon}.json',
        'target_col': f'target_{horizon}',
    }
    for horizon in ('h1', 'h2')
}
missing = [str(config[key]) for config in horizon_configs.values()
           for key in ('data_path', 'fold_path', 'tuning_path') if not config[key].exists()]
if missing:
    raise FileNotFoundError(
        'Task 1/2 artifacts are required. Confirm their saved Drive outputs:\n'
        + '\n'.join(missing)
    )
for directory in (output_dir, model_dir, drive_output_dir, drive_model_dir):
    if directory is not None:
        directory.mkdir(parents=True, exist_ok=True)
print(f'Repository: {repo_root}')
print(f'Active branch target: {REPO_BRANCH}')
print(f'Feature/fold inputs: {feature_dir}')
print(f'Frozen tuning inputs: {tuning_dir}')
print(f'Local results: {output_dir}; models: {model_dir}')
if IN_COLAB:
    print(f'Drive results: {drive_output_dir}; Drive models: {drive_model_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository: /content/AgriSence_model_training
Active branch target: feature/model-training
Feature/fold inputs: /content/drive/MyDrive/capstone-data/feature_engineering
Frozen tuning inputs: /content/drive/MyDrive/capstone-data/model_training
Local results: /content/AgriSence_model_training/results/artifacts; models: /content/AgriSence_model_training/results/artifacts/models
Drive results: /content/drive/MyDrive/capstone-data/final_evaluation; Drive models: /content/drive/MyDrive/capstone-data/models


<h3>2. Import the Shared Evaluation Workflow</h3>

<p>The next cell imports run_final_evaluation from agrisence.training.evaluate. That module loads the frozen artifacts, delegates model fitting and prediction to agrisence.training.ensemble.stacking, computes the result tables, and saves the outputs. All function implementations remain in the Python modules; none are defined in this notebook.</p>

In [2]:
import importlib
import sys

if globals().get('IN_COLAB', False):
    current_branch = subprocess.run(
        ['git', '-C', str(repo_root), 'branch', '--show-current'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current_branch != REPO_BRANCH:
        raise RuntimeError(f'Colab checkout is on {current_branch!r}; expected {REPO_BRANCH!r}.')
    tracked_changes = subprocess.run(
        ['git', '-C', str(repo_root), 'status', '--porcelain', '--untracked-files=no'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if tracked_changes:
        raise RuntimeError('Colab checkout has tracked edits. Preserve them before updating the repository.')

    github_token = os.getenv('GITHUB_TOKEN') or os.getenv('GH_TOKEN')
    if not github_token:
        from google.colab import userdata
        for secret_name in ('GITHUB_TOKEN', 'GH_TOKEN'):
            try:
                github_token = userdata.get(secret_name)
            except Exception:
                continue
            if github_token:
                break
    if not github_token:
        raise RuntimeError('Enable GITHUB_TOKEN or GH_TOKEN in Colab Secrets to update the private repository.')
    git_env = os.environ.copy()
    config_index = int(git_env.get('GIT_CONFIG_COUNT', '0'))
    git_env['GIT_CONFIG_COUNT'] = str(config_index + 1)
    git_env[f'GIT_CONFIG_KEY_{config_index}'] = 'http.https://github.com/.extraheader'
    git_env[f'GIT_CONFIG_VALUE_{config_index}'] = (
        'AUTHORIZATION: basic ' + base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    )
    try:
        subprocess.run(
            ['git', '-C', str(repo_root), 'fetch', 'origin', REPO_BRANCH],
            env=git_env, check=True,
        )
        subprocess.run(
            ['git', '-C', str(repo_root), 'merge', '--ff-only', f'origin/{REPO_BRANCH}'],
            check=True,
        )
    finally:
        del github_token, git_env
    revision = subprocess.run(
        ['git', '-C', str(repo_root), 'rev-parse', '--short', 'HEAD'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    print(f'Colab checkout updated: {REPO_BRANCH} at {revision}')
    for module_name in list(sys.modules):
        if module_name == 'agrisence' or module_name.startswith('agrisence.'):
            del sys.modules[module_name]

importlib.invalidate_caches()
try:
    from agrisence.training.evaluate import run_final_evaluation
except ImportError as error:
    raise RuntimeError(
        f'Task 3 could not be imported from {repo_root}. '
        f'Confirm the commit defining run_final_evaluation was pushed to {REPO_BRANCH}. '
        f'Original import error: {error}'
    ) from error

Colab checkout updated: feature/model-training at 7a44c5b


<h3>3. Configure Evaluation Outputs</h3>

In [3]:
evaluation_outputs = {
    'output_dir': output_dir,
    'model_dir': model_dir,
    'drive_output_dir': drive_output_dir,
    'drive_model_dir': drive_model_dir,
}

<h3>4. Final Evaluation: H1</h3>

<p>This call refits with H1's frozen settings, performs one test pass, displays all five models' results, and saves results_h1.csv, test_predictions_h1.csv, evaluation_manifest_h1.json, and stacking_h1.pkl to the configured local/Drive destinations. Once run, H1 test results must not guide feature, parameter, or model changes.</p>


In [4]:
results_h1 = run_final_evaluation('h1', horizon_configs['h1'], **evaluation_outputs)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(results_h1)

,horizon,model,market,commodity,scope,n_rows,mae_log,rmse_log,mape_price_pct,mae_price,rmse_price
0,h1,Persistence,Kisumu,Beans,series,11,0.044086,0.064679,4.241094,3.822121,5.659944
1,h1,Persistence,Kisumu,Maize,series,11,0.097291,0.129763,9.614318,3.783485,5.018111
2,h1,Persistence,Mombasa,Beans,series,11,0.090015,0.142103,8.321503,5.792152,9.133868
3,h1,Persistence,Mombasa,Maize,series,11,0.066394,0.108149,6.137652,2.175556,3.478099
4,h1,Persistence,Nairobi,Beans,series,11,0.097051,0.118158,9.217224,7.323586,9.181521
5,h1,Persistence,Nairobi,Maize,series,11,0.099808,0.133406,9.393349,3.414869,4.743564
6,h1,Persistence,Nakuru,Beans,series,11,0.092038,0.128689,9.013782,6.818990,9.651426
7,h1,Persistence,Nakuru,Maize,series,11,0.111255,0.150318,10.536034,3.282525,4.325309
8,h1,Persistence,ALL,ALL,pooled,88,0.087242,0.124412,8.309369,4.551660,6.813600
9,h1,Linear Regression,Kisumu,Beans,series,11,0.055927,0.074698,5.352100,4.999684,6.711748


<h3>5. Final Evaluation: H2</h3>

<p>Run the next cell once for the direct two-month cumulative log-change target, using H2's frozen settings and eligible rows. This is a separate run so an H2 issue does not require repeating H1's closed evaluation.</p>

<p>The call performs the same refit, single-pass prediction, reporting, and save workflow, producing results_h2.csv, test_predictions_h2.csv, evaluation_manifest_h2.json, and stacking_h2.pkl.</p>


In [5]:
results_h2 = run_final_evaluation('h2', horizon_configs['h2'], **evaluation_outputs)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(results_h2)

,horizon,model,market,commodity,scope,n_rows,mae_log,rmse_log,mape_price_pct,mae_price,rmse_price
0,h2,Persistence,Kisumu,Beans,series,10,0.072303,0.096595,6.788706,6.202444,8.374283
1,h2,Persistence,Kisumu,Maize,series,10,0.174673,0.223443,16.992542,6.777167,8.630044
2,h2,Persistence,Mombasa,Beans,series,10,0.146491,0.191696,13.264220,9.551094,12.756148
3,h2,Persistence,Mombasa,Maize,series,10,0.106465,0.179462,9.225348,3.394944,5.708400
4,h2,Persistence,Nairobi,Beans,series,10,0.132324,0.173953,11.889298,10.214167,13.852407
5,h2,Persistence,Nairobi,Maize,series,10,0.138413,0.170854,12.628306,4.558900,5.610852
6,h2,Persistence,Nakuru,Beans,series,10,0.165711,0.195544,16.377380,12.245222,14.483229
7,h2,Persistence,Nakuru,Maize,series,10,0.204082,0.266728,18.233991,5.779000,7.310020
8,h2,Persistence,ALL,ALL,pooled,80,0.142558,0.192692,13.174974,7.340367,10.164168
9,h2,Linear Regression,Kisumu,Beans,series,10,0.096188,0.124597,8.904480,8.506854,11.097270


<h3>6. Save and Verify the Fitted Models</h3>

<p>The evaluation workflow already saves the fitted models automatically. This explicit save-only step reloads the trusted H1/H2 bundles from local storage (or Drive if the local copy is missing), checks that their estimators are fitted, and saves/verifies both destinations without refitting or making predictions. It is safe to rerun independently of the closed test evaluations.</p>

<p>Each bundle contains the fitted Random Forest, XGBoost, Ridge meta-model, linear-regression pipeline, horizon, and feature-column order. Persistence needs no fitted estimator because it always predicts zero log change.</p>

<p>Local: results/artifacts/models/stacking_h1.pkl and stacking_h2.pkl.</p>

<p>Drive: MyDrive/capstone-data/models/stacking_h1.pkl and stacking_h2.pkl.</p>


In [6]:
import joblib
from sklearn.utils.validation import check_is_fitted

saved_models = []
for horizon in ('h1', 'h2'):
    model_filename = f'stacking_{horizon}.pkl'
    source_candidates = [model_dir / model_filename]
    if drive_model_dir is not None:
        source_candidates.append(drive_model_dir / model_filename)
    source_path = next((path for path in source_candidates if path.exists()), None)
    if source_path is None:
        raise FileNotFoundError(
            f'No saved {horizon.upper()} model bundle was found. '
            'Check the evaluation output paths; do not rerun a closed test evaluation.'
        )

    model_bundle = joblib.load(source_path)
    required_keys = {'rf_model', 'xgb_model', 'ridge_meta', 'linear_model', 'feature_cols', 'horizon'}
    missing_keys = required_keys.difference(model_bundle)
    if missing_keys or model_bundle['horizon'] != horizon:
        raise ValueError(f'Invalid {horizon.upper()} model bundle: missing keys={sorted(missing_keys)}')
    if not model_bundle['feature_cols']:
        raise ValueError(f'{horizon.upper()} feature order is missing.')
    for model_key in ('rf_model', 'xgb_model', 'ridge_meta', 'linear_model'):
        check_is_fitted(model_bundle[model_key])

    for destination, directory in (('Local', model_dir), ('Google Drive', drive_model_dir)):
        if directory is not None:
            directory.mkdir(parents=True, exist_ok=True)
            saved_path = directory / model_filename
            joblib.dump(model_bundle, saved_path)
            verified_bundle = joblib.load(saved_path)
            if verified_bundle['feature_cols'] != model_bundle['feature_cols']:
                raise ValueError(f'Feature order changed while saving {saved_path}')
            saved_models.append({
                'Horizon': horizon.upper(),
                'Destination': destination,
                'Model bundle': str(saved_path),
                'Features': len(verified_bundle['feature_cols']),
                'Size (bytes)': saved_path.stat().st_size,
            })

with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(saved_models))

,Horizon,Destination,Model bundle,Features,Size (bytes)
0,H1,Local,/content/AgriSence_model_training/results/artifacts/models/stacking_h1.pkl,14,1666351
1,H1,Google Drive,/content/drive/MyDrive/capstone-data/models/stacking_h1.pkl,14,1666351
2,H2,Local,/content/AgriSence_model_training/results/artifacts/models/stacking_h2.pkl,14,1209311
3,H2,Google Drive,/content/drive/MyDrive/capstone-data/models/stacking_h2.pkl,14,1209311
